# 🧠 Local Macro-Vector Generation — No API, No Cost

OpenRouter's free tier rate-limits you (~50 calls/day), so the API swarm fails.
This notebook builds the **same `macro_vectors.npz`** the main pipeline consumes,
but **entirely locally on the Kaggle GPU** — free, unlimited, no API key.

Two approaches (run both, compare, pick one):
1. **Local LLM swarm** — a small open model (Qwen2.5-3B-Instruct) role-plays the 5 personas.
2. **FinBERT** — a finance-tuned sentiment model; sentiment is fused into the macro vector.

Outputs (in `/kaggle/working`):
`macro_vectors_swarm.npz`, `macro_vectors_finbert.npz` (+ `*_meta.csv` for inspection).

**Settings:** Accelerator = GPU (T4), Internet = ON (to download the models from HuggingFace).

In [ ]:
!pip install -q transformers accelerate sentence-transformers sentencepiece

In [ ]:
import os, re, json, gc, glob, time
import numpy as np, pandas as pd, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print("Device:", DEVICE, "| dtype:", DTYPE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ── Config ────────────────────────────────────────────────────────────────────
TICKER   = "aapl"
N_EVENTS = 30                                  # how many distinct headlines to encode
OUT_DIR  = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

# Find the news CSV produced by NewsScrapper/newfetcher2.py (searches Kaggle inputs)
_search = []
for base in ("/kaggle/input", "data/raw", "."):
    _search += glob.glob(os.path.join(base, "**", f"polygon_{TICKER}_news.csv"), recursive=True)
    _search += glob.glob(os.path.join(base, "**", f"finnhub_{TICKER}_news.csv"), recursive=True)
NEWS_CSV = next((p for p in _search if os.path.exists(p)), None)
print("News CSV:", NEWS_CSV)

In [ ]:
# ── Load headlines ────────────────────────────────────────────────────────────
if NEWS_CSV:
    df  = pd.read_csv(NEWS_CSV)
    col = "text" if "text" in df.columns else ("title" if "title" in df.columns else df.columns[-1])
    step   = max(1, len(df) // N_EVENTS)
    events = df[col].dropna().astype(str).iloc[::step].tolist()[:N_EVENTS]
    print(f"Loaded {len(events)} headlines from column '{col}'")
else:
    print("No news CSV found — using a few synthetic events.")
    events = [
        "Apple beats earnings and raises guidance; shares jump after hours.",
        "Regulators open antitrust probe into Apple's App Store practices.",
        "Broad tech selloff as yields spike on hot inflation print.",
        "Warren Buffett trims Apple stake, citing valuation concerns.",
        "Quiet, range-bound session with light volume ahead of a holiday.",
    ]
for e in events[:3]:
    print(" •", e[:90])

In [ ]:
# ── Shared sentence encoder (produces the 384-D macro vectors) ────────────────
from sentence_transformers import SentenceTransformer
print("Loading all-MiniLM-L6-v2 ...")
encoder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
print("Embedding dim:", encoder.get_sentence_embedding_dimension())

## Approach 1 — Local LLM Swarm (Qwen2.5-3B-Instruct)

The same 5-persona swarm as the API version, but the model runs on the GPU. The 5
personas for one headline are generated in a single batched call for speed. To go
faster, switch `LLM_MODEL` to `Qwen/Qwen2.5-1.5B-Instruct`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"   # open, fits T4. Use -1.5B-Instruct for speed.
print("Loading", LLM_MODEL, "...")
tok = AutoTokenizer.from_pretrained(LLM_MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE).to(DEVICE).eval()

PERSONAS = {
    "momentum":       "You are a momentum trader. Predict if the initial price move continues.",
    "mean_reversion": "You are a mean-reversion trader. Predict if the initial price move fades.",
    "macro_risk":     "You are a macro risk analyst. Assess SYSTEMIC RISK flows.",
    "liquidity":      "You are a liquidity analyst. Predict the BID-ASK SPREAD effect ('up'=wider).",
    "volatility":     "You are a volatility trader. Predict the REALIZED VOLATILITY effect.",
}
JSON_RULE = (' Respond ONLY with a JSON object: {"direction":"up"|"down"|"neutral",'
             '"magnitude":0.0-1.0,"confidence":0.0-1.0,"reasoning":"..."}')

def _parse_json(t):
    t = re.sub(r"```(?:json)?\s*", "", t).strip().rstrip("`").strip()
    m = re.search(r"\{.*\}", t, re.DOTALL)
    try:
        return json.loads(m.group(0) if m else t)
    except Exception:
        return None

@torch.no_grad()
def swarm_event(event):
    prompts = []
    for sysmsg in PERSONAS.values():
        msgs = [{"role": "system", "content": sysmsg + JSON_RULE},
                {"role": "user",   "content": f"Analyze:\n\n{event}"}]
        prompts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    enc = tok(prompts, return_tensors="pt", padding=True).to(DEVICE)
    out = llm.generate(**enc, max_new_tokens=160, do_sample=False, pad_token_id=tok.pad_token_id)
    gens = tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    dir_map = {"up": 1.0, "down": -1.0, "neutral": 0.0}
    w, c, reasons, ok = 0.0, 0.0, [], 0
    for name, g in zip(PERSONAS, gens):
        p = _parse_json(g)
        if not p:
            continue
        ok += 1
        conf = float(p.get("confidence", 0.5))
        d = dir_map.get(str(p.get("direction", "neutral")).lower(), 0.0)
        w += d * conf; c += conf
        reasons.append(f"[{name}]: {p.get('reasoning', '')}")
    return " ".join(reasons), (w / c if c > 0 else 0.0), ok

vecs, meta, t0 = [], [], time.time()
for i, ev in enumerate(events):
    reasoning, direction, ok = swarm_event(ev)
    emb = encoder.encode(reasoning or ev, convert_to_numpy=True)
    vecs.append(emb); meta.append({"event": ev[:80], "direction": round(direction, 3), "ok": ok})
    print(f"  {i+1:2}/{len(events)}  ok={ok}/5  dir={direction:+.2f}  {ev[:50]}...")

vecs = np.array(vecs, dtype=np.float32)
np.savez(os.path.join(OUT_DIR, "macro_vectors_swarm.npz"), embeddings=vecs)
pd.DataFrame(meta).to_csv(os.path.join(OUT_DIR, "macro_swarm_meta.csv"), index=False)
print(f"\n✅ Saved macro_vectors_swarm.npz  shape={vecs.shape}  in {time.time()-t0:.0f}s")

# Free the LLM before loading FinBERT
del llm; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()

## Approach 2 — FinBERT (finance-tuned sentiment)

`ProsusAI/finbert` classifies each headline as positive / negative / neutral. We fuse
that sentiment into the macro vector by appending 4 sentiment dims to a 380-D headline
embedding (so the sentiment changes the vector *direction* and survives the env's
L2-normalization). Tiny model, runs in seconds.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

FIN = "ProsusAI/finbert"
print("Loading", FIN, "...")
ftok   = AutoTokenizer.from_pretrained(FIN)
fmodel = AutoModelForSequenceClassification.from_pretrained(FIN).to(DEVICE).eval()
id2label = {int(k): v for k, v in fmodel.config.id2label.items()}
print("Labels:", id2label)

@torch.no_grad()
def finbert_probs(texts, bs=16):
    out = []
    for i in range(0, len(texts), bs):
        enc = ftok(texts[i:i+bs], return_tensors="pt", padding=True,
                   truncation=True, max_length=256).to(DEVICE)
        out.append(F.softmax(fmodel(**enc).logits, dim=-1).cpu().numpy())
    return np.concatenate(out, 0)

P   = finbert_probs(events)                       # (N, 3)
lab = {v.lower(): k for k, v in id2label.items()}
p_pos = P[:, lab.get("positive", 0)]
p_neg = P[:, lab.get("negative", 1)]
p_neu = P[:, lab.get("neutral", 2)]
polarity = p_pos - p_neg                          # ∈ [-1, 1]

# 384-D macro vector = [380-D headline embedding | 4 sentiment dims]
emb       = encoder.encode(events, convert_to_numpy=True)          # (N, 384)
SENT_W    = 0.15                                                   # sentiment weight
sent_feat = np.stack([polarity, p_pos, p_neg, p_neu], 1) * SENT_W  # (N, 4)
macro     = np.concatenate([emb[:, :380], sent_feat], 1).astype(np.float32)

np.savez(os.path.join(OUT_DIR, "macro_vectors_finbert.npz"), embeddings=macro)
pd.DataFrame({"event": [e[:80] for e in events],
              "sentiment": [id2label[int(a)].lower() for a in P.argmax(1)],
              "polarity": polarity.round(3)}).to_csv(
    os.path.join(OUT_DIR, "macro_finbert_meta.csv"), index=False)
print(f"✅ Saved macro_vectors_finbert.npz  shape={macro.shape}")
print(pd.read_csv(os.path.join(OUT_DIR, 'macro_finbert_meta.csv')).head(8).to_string(index=False))

## Compare & next steps

In [ ]:
# ── Diversity sanity check (lower mean |cosine| = vectors are more distinct) ───
for f in ["macro_vectors_swarm.npz", "macro_vectors_finbert.npz"]:
    p = os.path.join(OUT_DIR, f)
    if os.path.exists(p):
        v  = np.load(p)["embeddings"]
        vn = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8)
        sim = vn @ vn.T
        off = sim[~np.eye(len(v), dtype=bool)]
        print(f"{f:30} shape={v.shape}  mean|cos|={np.abs(off).mean():.3f}")

### How to use these in the main pipeline (`drl_trading_pipeline.ipynb`)

Pick one `.npz` and point the env at it. Two easy ways:

**Same Kaggle session** — if you run this notebook then the main one in the same kernel,
the files are already in `/kaggle/working`. In the main notebook, **leave
`OPENROUTER_API_KEY = ""`** (so the API swarm cell is skipped), then add **one line after
that swarm cell**:
```python
MACRO_VECTORS_PATH = "/kaggle/working/macro_vectors_swarm.npz"   # or _finbert.npz
```

**Separate run** — download the chosen `.npz`, add it to your Kaggle dataset, and set
`MACRO_VECTORS_PATH` in the main notebook's config to that path.

Once you've decided which one trains better, I can wire it straight into the main
notebook's swarm cell as a drop-in replacement so it's automatic.